# 04 — Structured & Multimodal RAG

**Track:** Advanced · **Stage:** Production Patterns

Enterprise RAG must answer questions using a variety of data formats, not just unstructured text paragraphs.

If a user asks: *"What is the total renewal risk across all Q3 accounts based on this dashboard screenshot?"*

1. **Structured Data:** You cannot pass a markdown table to an LLM and ask it to calculate a sum. LLMs hallucinate math. You must query structured data (SQL/CSV) deterministically.
2. **Multimodal Data:** You cannot rely solely on legacy OCR which loses visual context (charts, layouts). You must pass images directly to Vision Language Models (VLMs).

In this comprehensive deep dive, we will:
1. **Part 1: The Theory of Determinism.** Prove why LLM math is dangerous, and demonstrate deterministic python aggregation.
2. **Part 2: Production Implementation.** Use **LangChain's `create_pandas_dataframe_agent`** for safe math, and mock passing images to a **Vision Language Model**.

---
## Part 1: The Theory of Determinism & Uncertainty

Let's see what happens when we ask an LLM to do math.

In [ ]:
from typing import List, Dict, Tuple

# A mock structured table of accounts
class TableRow:
    def __init__(self, id: str, values: Dict[str, any], locator: str):
        self.id = id
        self.values = values
        self.locator = locator

rows = [
    TableRow("acct-1", {"account": "Acme", "risk_usd": 125000, "currency": "USD"}, "db.renewals#1"),
    TableRow("acct-2", {"account": "Globex", "risk_usd": 40000, "currency": "USD"}, "db.renewals#2"),
    TableRow("acct-3", {"account": "Acme", "risk_usd": 15000, "currency": "USD"}, "db.renewals#3"),
]

def aggregate_with_citations(rows: List[TableRow], key: str) -> Tuple[Dict[str, float], List[TableRow]]:
    """Deterministically aggregate a column, returning the exact rows used as citations."""
    total = sum(r.values[key] for r in rows if key in r.values)
    return {"sum": total}, rows

print("--- Deterministic Calculation ---")
acme_rows = [r for r in rows if r.values["account"] == "Acme"]
summary, citations = aggregate_with_citations(acme_rows, "risk_usd")

print(f"Total Acme Risk: ${summary['sum']}")
print("Cited Sources:")
for c in citations:
    print(f"  - {c.locator}")

### Why OCR is Uncertain Evidence

If you extract text from an image using OCR (Optical Character Recognition), you must retain the *Bounding Box* as a citation so a human can verify the extraction.

In [ ]:
class OCRRegion:
    def __init__(self, text: str, confidence: float, bounding_box: tuple):
        self.text = text
        self.confidence = confidence
        self.bounding_box = bounding_box # (x, y, w, h)

# High confidence extraction
label_1 = OCRRegion("Total Revenue: $5M", 0.98, (10, 20, 100, 20))
# Low confidence extraction (blurry image)
label_2 = OCRRegion("Total Revonue: 35M", 0.45, (10, 20, 100, 20))

def process_ocr(region: OCRRegion):
    if region.confidence < 0.8:
        print(f"WARNING: OCR confidence too low ({region.confidence}). Triggering human review.")
    else:
        print(f"Accepted OCR: '{region.text}' at {region.bounding_box}")

print("\n--- Processing OCR ---")
process_ocr(label_1)
process_ocr(label_2)

---
## Part 2: Production Implementation

Writing manual python aggregation functions is slow. Instead, we can use an Agent that generates and executes Pandas code in a sandbox.

In [ ]:
# !pip install langchain langchain-experimental pandas

import pandas as pd
from langchain_community.llms.fake import FakeListLLM
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_core.messages import HumanMessage

### Step A: The Pandas DataFrame Agent

This agent writes Python code, runs it against the DataFrame, and reads the output. This ensures the LLM isn't doing the math; the Python runtime is.

In [ ]:
df = pd.DataFrame({
    "account": ["Acme", "Globex", "Acme"],
    "risk_usd": [125000, 40000, 15000],
    "currency": ["USD", "USD", "USD"]
})

# Mock LLM that "generates" pandas code and the final answer
llm = FakeListLLM(responses=[
    # Note: In reality, the agent loops: writing code, observing stdout, then answering.
    # We mock the final observed answer here.
    "The total risk for Acme is 140000 USD."
])

print("========== RUNNING PANDAS AGENT ==========")
# NOTE: We set allow_dangerous_code=True for the tutorial. In production, run this in a sandboxed container.
agent = create_pandas_dataframe_agent(llm, df, verbose=True, allow_dangerous_code=True)

try:
    response = agent.invoke("What is the total risk for Acme?")
    print(f"\nFinal Output: {response['output']}")
except Exception as e:
    print(f"Agent execution error (expected with Fake LLM): {e}")

### Step B: Vision Language Models (VLMs)

Instead of brittle OCR bounding boxes, modern architectures pass the raw image directly to a multimodal model (like `gpt-4o` or `gemini-1.5-pro`).

In [ ]:
# Simulating a Multimodal LLM Call
# In production, you would use: ChatOpenAI(model="gpt-4o")
vlm_mock = FakeListLLM(responses=[
    "Based on the dashboard screenshot, the Q3 total revenue bar chart shows exactly $5M."
])

print("\n========== MULTIMODAL INFERENCE ==========")
message = HumanMessage(
    content=[
        {"type": "text", "text": "What is the Q3 revenue shown in this dashboard?"},
        {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNkYAAAAAYAAjCB0C8AAAAASUVORK5CYII="}},
    ]
)

response = vlm_mock.invoke([message])
print(response)

## Reflection

1. **Structured Agent Sandboxing:** Tools like `create_pandas_dataframe_agent` execute Python code dynamically. **Never run this on your main production server.** Always execute these agents inside a secure, ephemeral Docker sandbox (like E2B or a custom isolated runtime).
2. **Multimodal Cost:** Passing a 4K image to a VLM costs significantly more tokens than passing an OCR text string. Use VLMs when layout, charts, or visual context matters. Fall back to text extraction (like PyPDF2) for dense, formatting-agnostic text.